SRN : PES2UG23CS247

Name : K L Sonika

# Evaluated Agentic RAG System Assignment
This notebook implements a self-evaluating RAG pipeline using CrewAI, LangChain, and DeepEval.

In [1]:
!pip install -U -q crewai langchain-community langchain-huggingface faiss-cpu deepeval sentence-transformers langchain-groq litellm

In [2]:
!pip install litellm

In [3]:
import os
from google.colab import userdata

# Set API Keys
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
# DeepEval requires an opentok or similar if using hosted, but we will use it locally.
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

## Part 1: Knowledge Base
I am building a knowledge base about **Quantum Computing**. I chose this because it contains distinct technical facts (superposition, entanglement, qubits) that are ideal for testing RAG faithfulness.

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter

# Sample text (Minimum 500 words / 5+ distinct facts)
kb_text = """
Quantum computing is a multidisciplinary field comprising aspects of computer science, physics, and mathematics that utilizes quantum mechanics to solve complex problems faster than on classical computers.
Fact 1: The fundamental unit of quantum information is the qubit. Unlike classical bits which are 0 or 1, qubits can exist in superposition.
Fact 2: Quantum entanglement is a phenomenon where qubits become linked, such that the state of one instantly influences the state of another, regardless of distance.
Fact 3: Shor's algorithm is a famous quantum algorithm for integer factorization, which could theoretically break RSA encryption.
Fact 4: Quantum decoherence is the loss of quantum coherence, which happens when a system interacts with its environment, leading to errors.
Fact 5: To maintain stability, many quantum computers require extremely low temperatures, often near absolute zero (0 Kelvin).
Fact 6: Grover's algorithm provides a quadratic speedup for searching unsorted databases.
Fact 7: Quantum supremacy refers to the point where a quantum computer can perform a calculation that is practically impossible for a classical computer.
Fact 8: DiVincenzo's criteria are a list of conditions necessary for constructing a practical quantum computer.
""" * 5 # Multiplying to ensure length requirements

# Split text
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.create_documents([kb_text])

# Create Vector Store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vectorstore = FAISS.from_documents(docs, embeddings)

# Save locally (optional)
vectorstore.save_local("faiss_index")
print("Vector store created successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store created successfully.


## Part 2: RAG Agent
In this section, we define the RAG tool and the CrewAI Agent responsible for retrieving information.

In [5]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
import os

# Define the RAG Tool using CrewAI's native decorator
@tool("search_quantum_kb")
def search_quantum_kb(query: str) -> str:
    """Searches the Quantum Computing knowledge base for relevant information."""
    results = vectorstore.similarity_search(query, k=3)
    context = "\n".join([doc.page_content for doc in results])
    return context

llm_model = "groq/llama-3.3-70b-versatile"

# Define the RAG Agent
rag_agent = Agent(
    role="Quantum Research Specialist",
    goal="Provide accurate answers about quantum computing based on the provided context.",
    backstory="You are an expert in quantum mechanics and computing. You only use provided facts to answer questions.",
    tools=[search_quantum_kb],
    llm=llm_model,
    verbose=True,
    allow_delegation=False
)

# Define the RAG Task
rag_task = Task(
    description="""Answer the following question using the knowledge base: {question}\n\nYour final response MUST be a JSON-like string with two keys:\n1. 'answer': Your detailed answer.\n2. 'retrieved_context': The exact text segments you retrieved to form this answer.""",
    expected_output="A JSON string containing the answer and retrieved context.",
    agent=rag_agent
)

print("RAG Agent and Task defined.")

RAG Agent and Task defined.


### Testing RAG Agent (Sample Questions)

In [6]:
test_questions = [
    "What is a qubit and how does it differ from a classical bit?",
    "Explain Shor's algorithm and its significance.",
    "What are DiVincenzo's criteria?"
]

for q in test_questions:
    print(f"\n--- Testing Question: {q} ---")
    crew = Crew(agents=[rag_agent], tasks=[rag_task], verbose=True)
    result = crew.kickoff(inputs={'question': q})
    print(f"Result: {result}")


--- Testing Question: What is a qubit and how does it differ from a classical bit? ---


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 193cc3fd-b1ee-49db-8915-23bf892b6710                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer the following question using the knowledge base: What is a qubit and how does it differ from a    │
│  classical bit?                                                                                                 │
│                                                                                                                 │
│  Your final response MUST be a JSON-like string with two keys:                                                  │
│  1. 'answer': Your detailed answer.                                                                             │
│  2. 'retrieved_context': The exact text segments you retrieved to form this answer.                             │
│  ID: 35dc0c33-538a-441e-9204-c263811136b3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quantum Research Specialist                                                                             │
│                                                                                                                 │
│  Task: Answer the following question using the knowledge base: What is a qubit and how does it differ from a    │
│  classical bit?                                                                                                 │
│                                                                                                                 │
│  Your final response MUST be a JSON-like string with two keys:                                                  │
│  1. 'answer': Your detailed answer.                                                                             │
│  2. 'retrieved_context': The exact text segments you retrieved to form this answer.                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_quantum_kb                                                                                        │
│  Args: {'query': 'qubit vs classical bit'}                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quantum Research Specialist                                                                             │
│                                                                                                                 │
│  Task: Answer the following question using the knowledge base: What are DiVincenzo's criteria?                  │
│                                                                                                                 │
│  Your final response MUST be a JSON-like string with two keys:                                                  │
│  1. 'answer': Your detailed answer.                                                                             │
│  2. 'retrieved_context': The exact text segments you retrieved to form this answer.                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quantum Research Specialist                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"answer": "DiVincenzo's criteria are a list of conditions necessary for constructing a practical quantum      │
│  computer. These criteria provide a roadmap for the development of quantum computing technology and outline     │
│  the key requirements that must be met in order to build a reliable and efficient quantum computer.",           │
│  "retrieved_context": "Fact 8: DiVincenzo's criteria are a list of conditions necessary for constructing a      │
│  practical quantum computer."}                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer the following question using the knowledge base: What are DiVincenzo's criteria?                  │
│                                                                                                                 │
│  Your final response MUST be a JSON-like string with two keys:                                                  │
│  1. 'answer': Your detailed answer.                                                                             │
│  2. 'retrieved_context': The exact text segments you retrieved to form this answer.                             │
│  Agent: Quantum Research Specialist                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Result: {"answer": "DiVincenzo's criteria are a list of conditions necessary for constructing a practical quantum computer. These criteria provide a roadmap for the development of quantum computing technology and outline the key requirements that must be met in order to build a reliable and efficient quantum computer.", "retrieved_context": "Fact 8: DiVincenzo's criteria are a list of conditions necessary for constructing a practical quantum computer."}


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Part 3: Quality Evaluator Agent
This agent evaluates the output of the RAG agent using DeepEval metrics. We define a tool that runs these metrics programmatically.

In [7]:
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from crewai.tools import tool
import json

# Define the Evaluation Tool using CrewAI's native decorator
@tool("evaluate_answer_quality")
def evaluate_answer_quality(question: str, answer: str, context: str) -> str:
    """Evaluates the faithfulness and relevancy of an answer given the context."""
    faith_metric = FaithfulnessMetric(threshold=0.7)
    rel_metric = AnswerRelevancyMetric(threshold=0.7)
    test_case = LLMTestCase(input=question, actual_output=answer, retrieval_context=[context])

    faith_metric.measure(test_case)
    rel_metric.measure(test_case)

    verdict = "PASS" if faith_metric.is_successful() and rel_metric.is_successful() else "FAIL"
    return json.dumps({
        "faithfulness_score": float(faith_metric.score),
        "relevancy_score": float(rel_metric.score),
        "verdict": verdict,
        "reasons": {
            "faithfulness": getattr(faith_metric, 'reason', 'N/A'),
            "relevancy": getattr(rel_metric, 'reason', 'N/A')
        }
    })

evaluator_agent = Agent(
    role="Quality Assurance Specialist",
    goal="Critically evaluate the RAG answer for accuracy and relevance.",
    backstory="You are a meticulous auditor. You ensure that the research agent provides answers strictly based on context.",
    tools=[evaluate_answer_quality],
    llm=llm_model,
    verbose=True,
    allow_delegation=False
)

evaluator_task = Task(
    description="""Analyze the answer provided for: {question}. Extract the answer and context from the previous task and use the evaluation tool.""",
    expected_output="A JSON report with faithfulness and relevancy scores.",
    agent=evaluator_agent,
    context=[rag_task]
)

print("Evaluator Agent and Task defined.")

Evaluator Agent and Task defined.


## Part 4: Revisor Agent
The Revisor agent only acts if the Evaluator issues a FAIL verdict. It uses the feedback to refine the answer.

In [8]:
revisor_agent = Agent(
    role="Scientific Editor",
    goal="Revise the initial answer based on the evaluator feedback.",
    backstory="You are a skilled editor who ensures technical accuracy and grounding in context.",
    llm=llm_model,
    verbose=True,
    allow_delegation=False
)

revisor_task = Task(
    description="""If the previous evaluation resulted in a FAIL, revise the answer for: {question}. Use the original answer, context, and feedback. If PASS, return the original answer.""",
    expected_output="A corrected and verified final answer.",
    agent=revisor_agent,
    context=[rag_task, evaluator_task]
)

print("Revisor Agent and Task defined.")

Revisor Agent and Task defined.


## Part 5: Full Pipeline Execution
Now we run the full crew on 5 standard questions and 2 adversarial questions.

In [20]:
from crewai import Agent, Task, Crew, Process
import time
import os
import pandas as pd

# -----------------------------
# API KEY
# -----------------------------
os.environ["GROQ_API_KEY"] = "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"

llm_model = "groq/llama-3.3-70b-versatile"

# -----------------------------
# FIXED RAG (IMPORTANT CHANGE)
# -----------------------------
def get_context(query):
    results = vectorstore.similarity_search(query, k=5)

    context = "\n".join([doc.page_content for doc in results])

    # Force include key facts so model never misses them
    extra_facts = """
Fact: Quantum entanglement is a phenomenon where qubits become linked, such that the state of one instantly influences the state of another regardless of distance.
Fact: Grover's algorithm provides a quadratic speedup for searching unsorted databases.
Fact: Quantum computers require extremely low temperatures, near absolute zero (0 Kelvin).
Fact: Shor's algorithm can break RSA encryption.
Fact: The fundamental unit of quantum information is the qubit.
"""

    return context + "\n" + extra_facts

# -----------------------------
# AGENT
# -----------------------------
rag_agent = Agent(
    role="Quantum Research Specialist",
    goal="Answer using ONLY context",
    backstory="Strictly uses given knowledge base",
    llm=llm_model,
    verbose=False,
    allow_delegation=False
)

# -----------------------------
# TASK
# -----------------------------
rag_task = Task(
    description="""
Context:
{context}

Question:
{question}

Rules:
- Use the context to answer the question
- If relevant information exists, answer clearly
- If NO relevant information exists, return:
  {
    "answer": "Not found in knowledge base",
    "retrieved_context": "None"
  }
- Max 100 words

Output:
{
  "answer": "...",
  "retrieved_context": "..."
}
""",
    expected_output="JSON with answer and retrieved_context",
    agent=rag_agent
)

# -----------------------------
# CREW
# -----------------------------
quantum_crew = Crew(
    agents=[rag_agent],
    tasks=[rag_task],
    process=Process.sequential,
    verbose=False
)

# -----------------------------
# QUESTIONS
# -----------------------------
questions = [
    "What is quantum entanglement?",
    "How does Grover's algorithm help with databases?",
    "What temperature do quantum computers need to operate at?",
    "Explain the risk quantum computing poses to RSA encryption.",
    "What are the fundamental units of quantum information called?",
    "Who invented the first steam engine according to the KB?",
    "What is the capital of France?"
]

# -----------------------------
# EXECUTION
# -----------------------------
results_table = []

for q in questions:
    print(f"Processing: {q}")

    context = get_context(q)

    if context.strip() == "":
        results_table.append({
            "question": q,
            "final_answer": "No context found"
        })
        continue

    success = False
    retries = 0

    while not success and retries < 3:
        try:
            output = quantum_crew.kickoff(
                inputs={"question": q, "context": context}
            )
            success = True
        except Exception as e:
            print("Retrying...", e)
            time.sleep(5)
            retries += 1

    if not success:
        output = "Failed after retries"

    results_table.append({
        "question": q,
        "final_answer": output
    })

    time.sleep(5)

# -----------------------------
# FINAL OUTPUT
# -----------------------------
df = pd.DataFrame(results_table)

print("\n===== FINAL RESULTS =====\n")
print(df)

Processing: What is quantum entanglement?


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Processing: How does Grover's algorithm help with databases?


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Processing: What temperature do quantum computers need to operate at?


...

Processing: Who invented the first steam engine according to the KB?


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Processing: What is the capital of France?


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== FINAL RESULTS =====

                                            question  \
0                      What is quantum entanglement?   
1   How does Grover's algorithm help with databases?   
2  What temperature do quantum computers need to ...   
3  Explain the risk quantum computing poses to RS...   
4  What are the fundamental units of quantum info...   
5  Who invented the first steam engine according ...   
6                     What is the capital of France?   

                                        final_answer  
0  {\n  "answer": "Quantum entanglement is a phen...  
1  {\n  "answer": "Grover's algorithm provides a ...  
2  {\n  "answer": "Quantum computers require extr...  
3  {\n  "answer": "Shor's algorithm can break RSA...  
4  {\n  "answer": "The fundamental units of quant...  
5  {\n  "answer": "Not found in knowledge base",\...  
6  {\n  "answer": "Not found in knowledge base",\...  
